# DACSES Source-to-Target Transformations

**Purpose:** Transform legacy DACSES source tables (T-series) into the new DACSES V-series (VDEMO, VCASE, VCMEM, VSORD, VOBLE, VLSUP) using Microsoft Fabric Spark.

**Source:** Mapping rules from `POC Data Mapping Summary.xlsx`.

**Architecture:**
1. Source tables land in a Fabric Lakehouse (`SourceLH`) via Mirroring or Copy Job.
2. This notebook reads source Delta tables, applies all transformations, and writes V-series Delta tables to a target Lakehouse (`TargetLH`).
3. A `t_reference` table provides cross-walk lookups (e.g. `OPEN`→`O`).

**Conventions:**
- All conversion-date defaults use the parameter `CONVERSION_DATE`.
- Tables are written using overwrite mode for idempotency.
- Each section corresponds to one tab in the mapping spreadsheet.

## 1. Parameters & Setup

In [ ]:
# Parameters (overridden by Data Factory pipeline at runtime)
SOURCE_LAKEHOUSE = "SourceLH"          # Lakehouse holding mirrored T-series tables
TARGET_LAKEHOUSE = "TargetLH"          # Lakehouse holding V-series Delta tables
CONVERSION_DATE  = "2026-05-06"        # Cut-over date used by default-value rules
HIGH_DATE        = "9999-12-31"        # Open-ended end date sentinel
DELAWARE_FIPS    = "1000000"           # Default FIPS when none can be derived

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType, IntegerType, DecimalType, DateType

spark.conf.set("spark.sql.caseSensitive", "false")

def src(name: str):
    """Read a source Delta table from the source lakehouse."""
    return spark.read.table(f"{SOURCE_LAKEHOUSE}.{name}")

def write_target(df, name: str):
    """Persist a transformed DataFrame to the target lakehouse (overwrite)."""
    (df.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{TARGET_LAKEHOUSE}.{name}"))
    print(f"Wrote {df.count():>6} rows to {TARGET_LAKEHOUSE}.{name}")

# --- Target SQL Server 2022 type alignment ---------------------------------
# The downstream pipeline bulk-loads these Delta tables into DACSES_POC_Target.
# Cast key columns to the EXACT decimal precisions defined in the target DDL
# (TargetDB/DACESS_Target_DDL.sql) so the TDS bulk insert does not silently
# truncate or fail with an arithmetic-overflow error.
TARGET_TYPES = {
    "Case_IDNO":          DecimalType(6, 0),
    "MemberMci_IDNO":     DecimalType(10, 0),
    "INDIVIDUAL_IDNO":    DecimalType(8, 0),
    "MemberSsn_NUMB":     DecimalType(9, 0),
    "OrderSeq_NUMB":      DecimalType(2, 0),
    "ObligationSeq_NUMB": DecimalType(2, 0),
    "Order_IDNO":         DecimalType(15, 0),
}
# All amount columns end with _AMNT and map to decimal(11,2).
AMOUNT_PRECISION = DecimalType(11, 2)

def conform(df):
    """Cast key/amount columns to target SQL Server precisions."""
    for col_name, dtype in TARGET_TYPES.items():
        if col_name in df.columns:
            df = df.withColumn(col_name, F.col(col_name).cast(dtype))
    for c in df.columns:
        if c.endswith("_AMNT"):
            df = df.withColumn(c, F.col(c).cast(AMOUNT_PRECISION))
    return df

## 2. Reference / Cross-walk Helper

All code-translation lookups (e.g. `OPEN`→`O`) come from `t_reference` keyed by `(view_nam, field_nam, old_code)`.

In [ ]:
ref = src("t_reference").select(
    F.col("view_nam").alias("view"),
    F.col("field_nam").alias("field"),
    F.col("old_code"),
    F.col("new_code"),
).cache()

def xwalk(df, src_col: str, view: str, field: str, target_col: str, default=None):
    """Left-join t_reference for a given (view, field) and replace src_col with new_code.
    Returns df with `target_col` populated (and src_col left intact)."""
    sub = (ref.where((F.col("view") == view) & (F.col("field") == field))
              .select(F.col("old_code").alias("_xw_old"), F.col("new_code").alias("_xw_new")))
    out = (df.join(sub, F.rtrim(df[src_col]) == F.rtrim(sub["_xw_old"]), "left")
             .withColumn(target_col, F.coalesce(F.col("_xw_new"), F.lit(default)))
             .drop("_xw_old", "_xw_new"))
    return out

## 3. Load Source DataFrames

In [ ]:
T1043 = src("T1043_PART")
T1044 = src("T1044_CASE")
T1045 = src("T1045_AP_EXT")
T1005 = src("T1005_LS_LIC_MATCH")
T1018 = src("T1018_SS_DEBT")
T1038 = src("T1038_FTIN_ELIG")
T1048 = src("T1048_SUB_ACCT")
T1049 = src("T1049_COURT_ORDER")
T1050 = src("T1050_ORD_SUB_ACCT")
T1051 = src("T1051_PART_ADDRESS")
T1052 = src("T1052_ORD_CHLD_INC")
T1074 = src("T1074_EVENT")
T1082 = src("T1082_CASE_PART")
T1096 = src("T1096_PATERNITY")

## 4. VDEMO  –  Member Demographics

**Sources:** T1043 (core), T1045 (NCP physical descriptors), T1005 (driver licence), T1051 (mailing-address phone), T1096 (paternity birth state).

**Key rules:**
- Parse `T1043.NAM` (`LAST, FIRST MIDDLE`) into `Last_NAME`, `First_NAME`, `Middle_NAME`.
- Convert `T1045.HEIGHT` (`5 11 `) to compact `"511"`.
- `LicenseDriverNo_TEXT` from T1005 where `LIC_AGENCY_CD='DMV'` and `LIC_TYPE_CD` in (`'D'`,`'DRV'`).
- `HomePhone_NUMB` from T1051 where `ADD_TYP_CD='MAIL'`.
- `BirthState_CODE` prefers T1096.BIRTH_ST then T1045.BIRTH_STATE.

In [ ]:
# Driver-license lookup (one row per MCI)
dl = (T1005
      .where((F.col("LIC_AGENCY_CD") == "DMV") & (F.col("LIC_TYPE_CD").isin("D", "DRV")))
      .groupBy("NCP_MCI_NUM")
      .agg(F.first("LIC_NUM", ignorenulls=True).alias("LicenseDriverNo_TEXT")))

# Mail-address phone (one row per MCI)
mail_phone = (T1051
              .where(F.col("ADD_TYP_CD") == "MAIL")
              .groupBy("MCI_NUM")
              .agg(F.first("HSE_PHONE_NUM", ignorenulls=True).alias("HomePhone_NUMB")))

# Parse NAM 'LAST, FIRST MIDDLE'
name_parts = (T1043
  .withColumn("_nam",   F.trim(F.col("NAM")))
  .withColumn("_last",  F.trim(F.split(F.col("_nam"), ",").getItem(0)))
  .withColumn("_rest",  F.trim(F.split(F.col("_nam"), ",").getItem(1)))
  .withColumn("_first", F.split(F.col("_rest"), " ").getItem(0))
  .withColumn("_mid",   F.expr("trim(substring(_rest, length(_first)+1, 100))")))

# Compact height: '5 11 ' -> '511'
height_clean = F.regexp_replace(F.col("HEIGHT"), " ", "")

vdemo = (name_parts.alias("p")
    .join(T1045.alias("x"), F.col("p.MCI_NUM") == F.col("x.MCI_NUM"), "left")
    .join(T1096.alias("pat"), F.col("p.MCI_NUM") == F.col("pat.CHILD_MCI_NUM"), "left")
    .join(dl, F.col("p.MCI_NUM") == dl["NCP_MCI_NUM"], "left")
    .join(mail_phone, F.col("p.MCI_NUM") == mail_phone["MCI_NUM"], "left")
    .select(
        # MCI offset by +1,000,000 mirrors the dummy-data convention; production = identity
        (F.col("p.MCI_NUM").cast("long") + F.lit(1000000)).alias("MemberMci_IDNO"),
        (F.col("p.MCI_NUM").cast("long") * 10).alias("INDIVIDUAL_IDNO"),
        F.col("p._last").alias("Last_NAME"),
        F.col("p._first").alias("First_NAME"),
        F.col("p._mid").alias("Middle_NAME"),
        F.lit(None).cast("string").alias("Suffix_NAME"),
        F.lit(None).cast("string").alias("Title_NAME"),
        F.col("p._nam").alias("FullDisplay_NAME"),
        F.col("p.SEX_CD").alias("MemberSex_CODE"),
        F.col("p.SOC_SEC_NUM").alias("MemberSsn_NUMB"),
        F.col("p.BIRTH_DT").alias("Birth_DATE"),
        F.trim(F.col("x.BIRTH_CITY")).alias("BirthCity_NAME"),
        F.coalesce(F.col("pat.BIRTH_ST"), F.col("x.BIRTH_STATE")).alias("BirthState_CODE"),
        F.lit("US").alias("BirthCountry_CODE"),
        height_clean.alias("DescriptionHeight_TEXT"),
        F.col("x.WEIGHT").cast("string").alias("DescriptionWeightLbs_TEXT"),
        F.col("p.ETHNIC_CD").alias("_ethnic_raw"),
        F.trim(F.col("x.HAIR_COLOR")).alias("ColorHair_CODE"),
        F.trim(F.col("x.EYE_COLOR")).alias("ColorEyes_CODE"),
        F.lit("ENG").alias("Language_CODE"),
        F.col("HomePhone_NUMB"),
        F.lit(None).cast("long").alias("WorkPhone_NUMB"),
        F.lit(None).cast("long").alias("CellPhone_NUMB"),
        F.lit(None).cast("string").alias("CONTACT_EML"),
        F.to_date(F.lit(CONVERSION_DATE)).alias("BeginValidity_DATE"),
        F.lit("CONV").alias("WorkerUpdate_ID"),
        F.lit(0).cast("long").alias("TransactionEventSeq_NUMB"),
        F.current_timestamp().alias("Update_DTTM"),
        F.col("LicenseDriverNo_TEXT"),
    ))

# Apply Race_CODE crosswalk (WH→W, BL→B, HI→H)
vdemo = (xwalk(vdemo, "_ethnic_raw", "VDEMO", "Race_CODE", "Race_CODE", default="U")
         .drop("_ethnic_raw"))

write_target(conform(vdemo), "VDEMO")

## 5. VCASE  –  Cases

**Sources:** T1044 (core), T1038 (federal tax intercept), T1051 (mail address for `RespondInit_CODE`), T1074 (events for `GoodCause_DATE`, `NonCoop_DATE`).

**Key rules:**
- `StatusCase_CODE`, `TypeCase_CODE`, `RsnStatusCase_CODE` from cross-walks.
- `Intercept_CODE = 'I'` if any T1038 row has `TAX_INT_STS_CD in ('URRS','SUBM')`; else `'N'`.
- `RespondInit_CODE = 'I'` (initiating) if NCP has DE mailing address; `'R'` (responding) otherwise.
- `NonCoop_DATE` = T1074.CREATE_DT for first event of TYP_CD `UCNC`.
- `CaseCategory_CODE`: composite from `IVD_TYP_CD` (e.g. DIRP→`DP`, MAO→`MO`, AFDC/NFAF→`IV`).

In [ ]:
# Intercept rule
intercept = (T1038
  .where(F.col("TAX_INT_STS_CD").isin("URRS", "SUBM"))
  .select("CASE_NUM").distinct()
  .withColumn("_intercept", F.lit("I")))

# NonCoop date = earliest UCNC event
noncoop = (T1074
  .where(F.col("TYP_CD") == "UCNC")
  .groupBy("CASE_NUM")
  .agg(F.min("CREATE_DT").alias("NonCoop_DATE")))

# GoodCause date from earliest WPFA/UPFA event
goodcause = (T1074
  .where(F.col("TYP_CD").isin("WPFA", "UPFA"))
  .groupBy("CASE_NUM")
  .agg(F.min("CREATE_DT").alias("GoodCause_DATE")))

# RespondInit_CODE – use NCP's MAIL address CORR_FIPS_CD
ncp_per_case = (T1082.where(F.col("TYP_CD") == "AP  ").select(
                  F.col("CASE_NUM"), F.col("MCI_NUM").alias("NCP_MCI")))
ncp_mail = (ncp_per_case.alias("n")
            .join(T1051.where(F.col("ADD_TYP_CD") == "MAIL").alias("a"),
                  F.col("n.NCP_MCI") == F.col("a.MCI_NUM"), "left")
            .select("n.CASE_NUM",
                    F.when(F.col("a.STATE_ADR") == "DE", F.lit("I"))
                     .when(F.col("a.STATE_ADR").isNotNull(), F.lit("R"))
                     .otherwise(F.lit("I")).alias("RespondInit_CODE")))

# Compose VCASE
base = (T1044.alias("c")
  .join(intercept,  "CASE_NUM", "left")
  .join(noncoop,    "CASE_NUM", "left")
  .join(goodcause,  "CASE_NUM", "left")
  .join(ncp_mail,   "CASE_NUM", "left"))

# Cross-walks
base = xwalk(base, "STATUS_CD",  "VCASE", "StatusCase_CODE",    "StatusCase_CODE",    default="O")
base = xwalk(base, "IVD_TYP_CD", "VCASE", "TypeCase_CODE",      "TypeCase_CODE",      default="N")
base = xwalk(base, "CLOSING_CD", "VCASE", "RsnStatusCase_CODE", "RsnStatusCase_CODE", default="OP")

# CaseCategory_CODE composite rule (simplified per mapping spreadsheet)
case_category = (
    F.when(F.col("IVD_TYP_CD") == "DIRP", F.lit("DP"))
     .when(F.col("IVD_TYP_CD") == "MAO ", F.lit("MO"))
     .when(F.col("IVD_TYP_CD") == "FC  ", F.lit("FC"))
     .when(F.col("IVD_TYP_CD").isin("AFDC", "NFAF", "COLL", "NPA "), F.lit("IV"))
     .otherwise(F.lit("IV")))

vcase = base.select(
    F.col("CASE_NUM").cast("long").alias("Case_IDNO"),
    F.col("StatusCase_CODE"),
    F.col("TypeCase_CODE"),
    F.col("RsnStatusCase_CODE"),
    F.col("RespondInit_CODE"),
    F.lit("APP").alias("SourceRfrl_CODE"),
    F.col("STATUS_EFF_DT").alias("Opened_DATE"),
    F.col("PROC_STS_EFF_DT").alias("StatusCurrent_DATE"),
    F.col("COUNTY_CD").cast("int").alias("County_IDNO"),
    F.col("COUNTY_CD").cast("int").alias("Office_IDNO"),
    F.lpad(F.col("COUNTY_CD"), 7, "0").alias("AssignedFips_CODE"),
    F.when(F.col("GOOD_CAUSE_RSN_CD").isNotNull(), F.lit("Y")).otherwise("N").alias("GoodCause_CODE"),
    F.lit("N").alias("Restricted_INDC"),
    F.coalesce(F.col("_intercept"), F.lit("N")).alias("Intercept_CODE"),
    F.col("MED_SUP_ONLY_IND").alias("MedicalOnly_INDC"),
    F.lit("Y").alias("Jurisdiction_INDC"),
    F.lit("APP").alias("IvdApplicant_CODE"),
    F.col("APPL_REQ_DT").alias("AppReq_DATE"),
    F.col("APPL_RETURN_DT").alias("AppRetd_DATE"),
    F.lit("SPO").alias("CpRelationshipToNcp_CODE"),
    F.col("RESP_WORKER_ID").alias("Worker_ID"),
    F.col("STATUS_EFF_DT").alias("BeginValidity_DATE"),
    F.col("RESP_WORKER_ID").alias("WorkerUpdate_ID"),
    F.lit(0).cast("long").alias("TransactionEventSeq_NUMB"),
    F.current_timestamp().alias("Update_DTTM"),
    case_category.alias("CaseCategory_CODE"),
    F.lit("F").alias("ServiceRequested_CODE"),
    F.when(F.col("STATUS_CD") == "CLSD", "CLSD").otherwise("ACTV").alias("StatusEnforce_CODE"),
    F.col("NonCoop_DATE"),
    F.col("GoodCause_DATE"),
)

write_target(conform(vcase), "VCASE")

## 6. VCMEM  –  Case Members

**Sources:** T1082 (case-part roles), T1044 (PFA / family-violence), T1043 (SEX_CD for relationship to child).

**Key rules:**
- `CaseRelationship_CODE` cross-walk: AP→N, CP→C, DEP→D, OTHR→O.
- `CaseMemberStatus_CODE`: STATUS_CD direct (`A`/`I`).
- `FAMILYVIOLENCE_INDC = 'Y'` if T1044 has `GOOD_CAUSE_RSN_CD in ('WPFA','UPFA')` or `UNWORK_RSN_CD in ('WPFA','UPFA')`.

In [ ]:
fv = (T1044
  .where(F.col("GOOD_CAUSE_RSN_CD").isin("WPFA", "UPFA") |
         F.col("UNWORK_RSN_CD").isin("WPFA", "UPFA"))
  .select("CASE_NUM").distinct()
  .withColumn("_fv", F.lit("Y")))

vcmem_base = (T1082.alias("m")
  .join(T1043.alias("p"), F.col("m.MCI_NUM") == F.col("p.MCI_NUM"), "left")
  .join(fv,                 "CASE_NUM", "left"))

vcmem_base = xwalk(vcmem_base, "TYP_CD", "VCMEM", "CaseRelationship_CODE", "CaseRelationship_CODE", default="O")

vcmem = vcmem_base.select(
    F.col("CASE_NUM").cast("long").alias("Case_IDNO"),
    (F.col("m.MCI_NUM").cast("long") + F.lit(1000000)).alias("MemberMci_IDNO"),
    F.col("CaseRelationship_CODE"),
    F.col("STATUS_CD").alias("CaseMemberStatus_CODE"),
    F.when(F.col("TYP_CD") == "CP  ", F.when(F.col("p.SEX_CD") == "F", "MOT").otherwise("FAT"))
     .when(F.col("TYP_CD") == "DEP ", F.lit("MOT"))
     .otherwise(F.lit(None)).alias("CpRelationshipToChild_CODE"),
    F.when(F.col("TYP_CD") == "AP  ", F.lit("FAT"))
     .when(F.col("TYP_CD") == "DEP ", F.lit("FAT"))
     .otherwise(F.lit(None)).alias("NcpRelationshipToChild_CODE"),
    F.lit("N").alias("BenchWarrant_INDC"),
    F.when(F.col("TYP_CD") == "CP  ", "Y").otherwise("N").alias("Applicant_CODE"),
    F.to_date(F.lit(CONVERSION_DATE)).alias("BeginValidity_DATE"),
    F.lit("CONV").alias("WorkerUpdate_ID"),
    F.lit(0).cast("long").alias("TransactionEventSeq_NUMB"),
    F.current_timestamp().alias("Update_DTTM"),
    F.coalesce(F.col("_fv"), F.lit("N")).alias("FAMILYVIOLENCE_INDC"),
)

write_target(conform(vcmem), "VCMEM")

## 7. VSORD  –  Court Orders

**Sources:** T1049 (orders) and T1044 (file id).

**Key rules:**
- `OrderSeq_NUMB = T1049.BLIND_KEY`.
- `Order_IDNO` synthesized as `9 * 10^14 + (Case * 100 + Seq)` for demo.
- `File_ID = replace(T1044.FAM_COURT_FILE, '-', '')`.
- `IssuingOrderFips_CODE` derived from `T1049.ORIGINATING_ST_CD` (DE→1000300, PA→4200000, etc.).
- `TypeOrder_CODE`: `'M'` if `MOD_TYP_CD='MOD '`, else `'O'`.

In [ ]:
# Two-letter state → 5-digit FIPS state code (truncated demo set; extend as needed)
STATE_FIPS = {
    "DE": "1000300", "PA": "4200000", "NJ": "3400000", "MD": "2400000",
    "NY": "3600000", "VA": "5100000", "CA": "0600000", "TX": "4800000",
}
fips_expr = F.create_map([F.lit(x) for kv in STATE_FIPS.items() for x in kv])

vsord = (T1049.alias("o")
  .join(T1044.alias("c"), "CASE_NUM", "left")
  .select(
    F.col("o.CASE_NUM").cast("long").alias("Case_IDNO"),
    F.col("o.BLIND_KEY").cast("int").alias("OrderSeq_NUMB"),
    (F.lit(900000000000000) + (F.col("o.CASE_NUM").cast("long") * 100 + F.col("o.BLIND_KEY"))).alias("Order_IDNO"),
    F.regexp_replace(F.col("c.FAM_COURT_FILE"), "-", "").alias("File_ID"),
    F.col("o.COURT_DT").alias("OrderEnt_DATE"),
    F.col("o.COURT_DT").alias("OrderIssued_DATE"),
    F.col("o.ORDER_START_DT").alias("OrderEffective_DATE"),
    F.col("o.ORDER_END_DT").alias("OrderEnd_DATE"),
    F.lit("A").alias("StatusOrder_CODE"),
    F.col("o.COURT_DT").alias("StatusOrder_DATE"),
    F.col("o.INS_ORDERED_IND").alias("InsOrdered_CODE"),
    F.lit("N").alias("MedicalOnly_INDC"),
    F.col("o.MELSON_DEV_IND").alias("GuidelinesFollowed_INDC"),
    F.coalesce(fips_expr.getItem(F.col("o.ORIGINATING_ST_CD")), F.lit(DELAWARE_FIPS)).alias("IssuingOrderFips_CODE"),
    F.col("o.CREATE_WORKER_ID").alias("WorkerUpdate_ID"),
    F.col("o.CREATE_DT").alias("BeginValidity_DATE"),
    F.when(F.col("o.MOD_TYP_CD") == "MOD ", "M").otherwise("O").alias("TypeOrder_CODE"),
    F.when(F.col("c.IVD_TYP_CD") == "DIRP", "Y").otherwise("N").alias("DirectPay_INDC"),
    F.lit("C").alias("SourceOrdered_CODE"),
  ))

write_target(conform(vsord), "VSORD")

## 8. VOBLE  –  Obligations

**Sources:** T1049 (order), T1050 (ordered amounts), T1052 (dependents on order), T1048 (sub-account), T1051 (mail address FIPS), T1044 (skip rules).

**Key rules:**
- Skip if `T1044.IVD_TYP_CD='DIRP'`.
- `ObligationSeq_NUMB` = row_number per (Case, Order).
- `TypeDebt_CODE`: cross-walk from T1050.SUB_TYP_CD.
- `Fips_CODE`: T1048.URESA_CD when SUB_TYP_CD='OSTATA'; else T1051.PAYMENT_FIPS_CD where ADD_TYP_CD='MAIL'; else `1000000`.
- `FreqPeriodic_CODE`: cross-walk from T1049.CHRG_FREQ_CD (overridden by T1044.CHARGE_FREQ_CD when no-charge).
- `Periodic_AMNT`: T1050.FREQ_AMT for SPLSUP/CURSUP/MEDSUP, else 0.

In [ ]:
# Filter out DIRP cases (no VOBLE)
non_dirp = T1044.where(F.col("IVD_TYP_CD") != "DIRP").select("CASE_NUM").distinct()

# CP MCI per case (used as default check-recipient)
cp_per_case = (T1082.where(F.col("TYP_CD") == "CP  ")
                  .select(F.col("CASE_NUM"), F.col("MCI_NUM").alias("CP_MCI")))

# CP mail-address FIPS
cp_mail_fips = (cp_per_case.alias("c")
  .join(T1051.where(F.col("ADD_TYP_CD") == "MAIL").alias("a"),
        F.col("c.CP_MCI") == F.col("a.MCI_NUM"), "left")
  .select("c.CASE_NUM",
          F.lpad(F.col("a.PAYMENT_FIPS_CD").cast("string"), 7, "0").alias("_mail_fips")))

# OSTATA URESA per case (interstate)
ostata = (T1048.where(F.col("SUB_TYP_CD") == "OSTATA")
          .groupBy("CASE_NUM")
          .agg(F.first("URESA_CD", ignorenulls=True).alias("_ostata_uresa")))

# Build VOBLE skeleton: one row per T1050 line
voble_base = (T1050.alias("l")
  .join(non_dirp,                     "CASE_NUM", "inner")
  .join(T1049.alias("o"),
        (F.col("l.CASE_NUM") == F.col("o.CASE_NUM")) & (F.col("l.BLIND_KEY") == F.col("o.BLIND_KEY")), "left")
  .join(T1044.alias("c"),             F.col("l.CASE_NUM") == F.col("c.CASE_NUM"), "left")
  .join(cp_per_case.alias("cp"),      F.col("l.CASE_NUM") == F.col("cp.CASE_NUM"), "left")
  .join(cp_mail_fips,                 F.col("l.CASE_NUM") == cp_mail_fips["CASE_NUM"], "left")
  .join(ostata,                       F.col("l.CASE_NUM") == ostata["CASE_NUM"], "left"))

# Cross-walks
voble_base = xwalk(voble_base, "l.SUB_TYP_CD",   "VOBLE", "TypeDebt_CODE",     "TypeDebt_CODE",     default="OT")
voble_base = xwalk(voble_base, "o.CHRG_FREQ_CD", "VOBLE", "FreqPeriodic_CODE", "FreqPeriodic_CODE", default="M")

# ObligationSeq_NUMB - sequential within (Case, Order)
w = Window.partitionBy("l.CASE_NUM", "l.BLIND_KEY").orderBy("l.SUB_TYP_CD")
voble_base = voble_base.withColumn("ObligationSeq_NUMB", F.row_number().over(w))

# Fips_CODE rule
fips_rule = (
    F.when(F.col("l.SUB_TYP_CD") == "OSTATA",
           F.lpad(F.col("_ostata_uresa").cast("string"), 7, "0"))
     .when(F.col("_mail_fips").isNotNull() & (F.col("_mail_fips") != "0000000"),
           F.col("_mail_fips"))
     .otherwise(F.lit(DELAWARE_FIPS))
)

# Periodic_AMNT rule
periodic_amnt = (F.when(F.col("l.SUB_TYP_CD").isin("SPLSUP", "CURSUP", "MEDSUP", "CSCHLD"),
                       F.col("l.FREQ_AMT"))
                  .otherwise(F.lit(0).cast(DecimalType(11, 2))))

voble = voble_base.select(
    F.col("l.CASE_NUM").cast("long").alias("Case_IDNO"),
    F.col("l.BLIND_KEY").cast("int").alias("OrderSeq_NUMB"),
    F.col("ObligationSeq_NUMB").cast("int"),
    (F.col("cp.CP_MCI").cast("long") + F.lit(1000000)).alias("MemberMci_IDNO"),
    F.col("TypeDebt_CODE"),
    fips_rule.alias("Fips_CODE"),
    F.col("FreqPeriodic_CODE"),
    periodic_amnt.alias("Periodic_AMNT"),
    periodic_amnt.alias("ExpectToPay_AMNT"),
    F.lit("C").alias("ExpectToPay_CODE"),
    F.col("o.ORDER_START_DT").alias("BeginObligation_DATE"),
    F.coalesce(F.col("o.ORDER_END_DT"), F.lit(None).cast(DateType())).alias("EndObligation_DATE"),
    F.col("c.LAST_CHRG_DT").alias("AccrualLast_DATE"),
    F.col("c.NEXT_CHARGE_DT").alias("AccrualNext_DATE"),
    F.to_date(F.lit(CONVERSION_DATE)).alias("BeginValidity_DATE"),
)

write_target(conform(voble), "VOBLE")

## 9. VLSUP  –  Ledger / Support History

**Sources:** VOBLE (driving keys), T1018 (debt buckets by `SS_TYP_CD`), T1048 (sub-account balances).

**Key rules:**
- One row per VOBLE row.
- `OweTotPaa_AMNT` = sum T1018.BAL_DUE where SS_TYP_CD='PERMAN'.
- `OweTotUda_AMNT` = sum T1018.BAL_DUE where SS_TYP_CD='UNASSD'.
- `OweTotNaa_AMNT` = sum T1018.BAL_DUE for any other SS_TYP_CD.
- `OweTotIvef_AMNT` = sum T1048.BALANCE where SUB_TYP_CD='FCAREA'.
- `OweTotMedi_AMNT` = sum T1048.BALANCE where SUB_TYP_CD='MEDA'.
- `TypeWelfare_CODE`: `'A'` if T1044.IVD_TYP_CD='AFDC', `'M'` if MAO, else `'N'`.

In [ ]:
# T1018 buckets per case
t1018_buckets = (T1018.groupBy("CASE_NUM").agg(
    F.sum(F.when(F.rtrim(F.col("SS_TYP_CD")) == "PERMAN", F.col("BAL_DUE")).otherwise(0)).alias("OweTotPaa_AMNT"),
    F.sum(F.when(F.rtrim(F.col("SS_TYP_CD")) == "UNASSD", F.col("BAL_DUE")).otherwise(0)).alias("OweTotUda_AMNT"),
    F.sum(F.when(~F.rtrim(F.col("SS_TYP_CD")).isin("PERMAN", "UNASSD"), F.col("BAL_DUE")).otherwise(0)).alias("OweTotNaa_AMNT"),
    F.sum("BAL_DUE").alias("OweTotCurSup_AMNT"),
))

# T1048 buckets per case
t1048_buckets = (T1048.groupBy("CASE_NUM").agg(
    F.sum(F.when(F.rtrim(F.col("SUB_TYP_CD")) == "FCAREA", F.col("BALANCE").cast(DecimalType(13, 2))).otherwise(0)).alias("OweTotIvef_AMNT"),
    F.sum(F.when(F.rtrim(F.col("SUB_TYP_CD")) == "MEDA",   F.col("BALANCE").cast(DecimalType(13, 2))).otherwise(0)).alias("OweTotMedi_AMNT"),
))

# Type-of-welfare from T1044
tw = T1044.select(
    F.col("CASE_NUM"),
    F.when(F.col("IVD_TYP_CD") == "AFDC", "A")
     .when(F.col("IVD_TYP_CD") == "MAO ", "M")
     .otherwise("N").alias("TypeWelfare_CODE"),
)

vlsup = (voble.alias("v")
  .join(t1018_buckets, F.col("v.Case_IDNO") == t1018_buckets["CASE_NUM"], "left")
  .join(t1048_buckets, F.col("v.Case_IDNO") == t1048_buckets["CASE_NUM"], "left")
  .join(tw,            F.col("v.Case_IDNO") == tw["CASE_NUM"], "left")
  .select(
    F.col("v.Case_IDNO"),
    F.col("v.OrderSeq_NUMB"),
    F.col("v.ObligationSeq_NUMB"),
    F.date_format(F.current_date(), "yyyyMM").cast("int").alias("SupportYearMonth_NUMB"),
    F.col("TypeWelfare_CODE"),
    F.col("v.Periodic_AMNT").alias("TransactionCurSup_AMNT"),
    F.coalesce(F.col("OweTotCurSup_AMNT"), F.lit(0)).alias("OweTotCurSup_AMNT"),
    F.coalesce(F.col("OweTotCurSup_AMNT"), F.lit(0)).alias("AppTotCurSup_AMNT"),
    F.col("v.Periodic_AMNT").alias("MtdCurSupOwed_AMNT"),
    F.to_date(F.lit(CONVERSION_DATE)).alias("Batch_DATE"),
    F.lit("WGE").alias("SourceBatch_CODE"),
    F.to_date(F.lit(CONVERSION_DATE)).alias("Receipt_DATE"),
    F.to_date(F.lit(CONVERSION_DATE)).alias("Distribute_DATE"),
    F.lit(0).cast("long").alias("EventGlobalSeq_NUMB"),
    F.lit("D").alias("TYPERECORD_CODE"),
    # Bonus columns retained for downstream reconciliation
    F.coalesce(F.col("OweTotPaa_AMNT"),  F.lit(0)).alias("OweTotPaa_AMNT"),
    F.coalesce(F.col("OweTotUda_AMNT"),  F.lit(0)).alias("OweTotUda_AMNT"),
    F.coalesce(F.col("OweTotNaa_AMNT"),  F.lit(0)).alias("OweTotNaa_AMNT"),
    F.coalesce(F.col("OweTotIvef_AMNT"), F.lit(0)).alias("OweTotIvef_AMNT"),
    F.coalesce(F.col("OweTotMedi_AMNT"), F.lit(0)).alias("OweTotMedi_AMNT"),
  ))

write_target(conform(vlsup), "VLSUP")

## 10. Validation

Compare row counts to expected values from the dummy load.

In [ ]:
for tbl in ["VDEMO", "VCASE", "VCMEM", "VSORD", "VOBLE", "VLSUP"]:
    n = spark.read.table(f"{TARGET_LAKEHOUSE}.{tbl}").count()
    print(f"{tbl:<6} : {n:>4} rows")